# Exploring U.S. Census Data Across Seven MSAs with the IPUMS NHGIS API

## Introduction and objectives

Census data provide a fundamental source of information on the demographic, social, and economic characteristics of the U.S. population. This notebook provides a brief, reproducible guide to accessing and exploring census data at the Census Block Group (CBG) level using the IPUMS NHGIS API.

The notebook focuses on seven U.S. MSAs: Los Angeles–Long Beach–Anaheim (LA), Houston–Pasadena–The Woodlands (Houston), Atlanta–Sandy Springs–Roswell (Atlanta), Miami–Fort Lauderdale–West Palm Beach (Miami), Seattle–Tacoma–Bellevue (Seattle), Denver–Aurora–Centennial (Denver), and Minneapolis–Saint Paul (Twin Cities). These seven MSAs are the examples presented in the [Visitor Census paper](https://www.nature.com/articles/s41597-025-05410-0).

The workflow has three main objectives:
1. Extract **2020 CBG boundaries** from U.S. Census Bureau TIGER/Line shapefiles.
2. Extract **2021 ACS 5-year** total population and median household income using the **IPUMS NHGIS API**.
3. Link the attributes to CBG boundaries and explore their statistical and spatial distributions.

## Computational set-up

### Required packages

The workflow uses `requests` for API access, `pandas` for tabular data, `geopandas` for spatial data, and `matplotlib` for visualization. `folium` and `mapclassify` support interactive and classified maps.


In [ ]:
import sys, subprocess, importlib.util, requests, io, zipfile, time
import pandas as pd
import numpy as np

for pkg in ["geopandas", "matplotlib", "folium", "mapclassify"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

import geopandas as gpd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

### IPUMS NHGIS API set-up

Follow the IPUMS API instructions at https://developer.ipums.org/docs/v2/get-started/ to register for an API key. Replace `YOUR_IPUMS_API_KEY` below with your own key before running the API cells.


In [ ]:
API_KEY = "59cba10d8a5da536fc06b59d14cda3aececc4d36b30ac927d6a12778"
headers = {"Authorization": API_KEY}

url = "https://api.ipums.org/metadata/datasets"
params = {"collection": "nhgis", "version": 2}
response = requests.get(url, params=params, headers=headers)
response.raise_for_status()
data = response.json()

## Extract census boundaries from the U.S. Census Bureau TIGER/Line file download service

### CBG boundaries for seven target MSAs

We consider the following parameters when building the extraction function:

* `state_fips`: State FIPS code, e.g., `"27"` for Minnesota.
* `county_fips`: County FIPS code, e.g., `"003"` for Anoka County, Minnesota.
* `year`: Census year, here we use 2020.

The data extraction process may take a few minutes, depending on computing resources.

In [ ]:
def get_msa_cbg(state_fips, county_fips, year=2020):
    url = f"https://www2.census.gov/geo/tiger/TIGER{year}/BG/tl_{year}_{state_fips}_bg.zip"
    gdf = gpd.read_file(url)
    return gdf[gdf["COUNTYFP"].isin(county_fips)].copy()

TCMA_county_fips = ["003", "019", "037", "053", "123", "139", "163"]
LA_county_fips = ["037", "059"]
Houston_county_fips = ["015", "039", "071", "157", "167", "201", "291", "339", "407", "473"]
Seattle_county_fips = ["033", "053", "061"]
Miami_county_fips = ["011", "086", "099"]
Atlanta_county_fips = ["013", "015", "035", "045", "057", "063", "067", "077", "085", "089", "097", "113", "117", "121", "135", "143", "149", "151", "159", "187", "199", "211", "217", "223", "227", "231", "247", "255", "297"]
Denver_county_fips = ["001", "005", "014", "019", "031", "035", "039", "047", "059", "093"]

TCMA_cbg = get_msa_cbg("27", TCMA_county_fips)
LA_cbg = get_msa_cbg("06", LA_county_fips)
Houston_cbg = get_msa_cbg("48", Houston_county_fips)
Seattle_cbg = get_msa_cbg("53", Seattle_county_fips)
Miami_cbg = get_msa_cbg("12", Miami_county_fips)
Atlanta_cbg = get_msa_cbg("13", Atlanta_county_fips)
Denver_cbg = get_msa_cbg("08", Denver_county_fips)

### Some statistics and visualizations

Combine CBG boundaries for seven MSAs into a single GeoDataFrame and visualize their spatial extents interactively.

In [ ]:
msa_dict = {
    "TCMA": TCMA_cbg,
    "Los Angeles": LA_cbg,
    "Houston": Houston_cbg,
    "Seattle": Seattle_cbg,
    "Miami": Miami_cbg,
    "Atlanta": Atlanta_cbg,
    "Denver": Denver_cbg
}

all_msa_cbg = gpd.GeoDataFrame(
    pd.concat(
        [gdf.to_crs(TCMA_cbg.crs).assign(MSA=name) for name, gdf in msa_dict.items()],
        ignore_index=True
    ),
    geometry="geometry",
    crs=TCMA_cbg.crs
)

all_msa_cbg.explore(
    column="MSA",
    categorical=True,
    tooltip=["MSA", "GEOID"],
    #tiles="CartoDB positron"
)


Report the number of CBGs and map their boundaries for the seven MSAs.

In [ ]:
def summarize_msa_cbg(gdf, name="MSA", plot=True):
    print(f"{name}: {len(gdf):,} CBGs")
    print(f"Counties: {gdf['COUNTYFP'].nunique()}")
    if plot:
        ax = gdf.plot(figsize=(8, 8), color="lightblue", edgecolor="black", linewidth=0.2)
        ax.set_title(f"{name} Census Block Groups (n={len(gdf):,})")
        ax.axis("off")
    #return gdf[["STATEFP", "COUNTYFP", "GEOID"]]

In [ ]:
summarize_msa_cbg(TCMA_cbg, "Twin Cities")
summarize_msa_cbg(LA_cbg, "Los Angeles")
summarize_msa_cbg(Houston_cbg, "Houston")
summarize_msa_cbg(Seattle_cbg, "Seattle")
summarize_msa_cbg(Miami_cbg, "Miami")
summarize_msa_cbg(Atlanta_cbg, "Atlanta")
summarize_msa_cbg(Denver_cbg, "Denver")

## Extract 2021 ACS CBG Attributes from IPUMS NHGIS API

### Data extraction

The extraction process includes four main steps:

1. **Identify the NHGIS dataset and tables** for the 2021 ACS 5-year estimates, including total population and median household income.
2. **Submit an NHGIS extract request** using the IPUMS API key, specifying block groups as the geographic level and the states covering the seven MSAs.
3. **Check the extract status and download the completed data** from IPUMS.
4. **Filter the downloaded CBG records by state and county FIPS codes** to obtain the ACS attributes for each MSA.

1. **Identify the NHGIS dataset and tables** for the 2021 ACS 5-year estimates, including total population and median household income.

In [ ]:
headers = {"Authorization": API_KEY}
dataset = "2017_2021_ACS5a" # 2021 ACS 5-year estimates

# Check table metadata
meta_url = f"https://api.ipums.org/metadata/datasets/{dataset}"
meta = requests.get(meta_url, headers=headers, params={"collection": "nhgis", "version": 2}).json()

tables = pd.DataFrame(meta["dataTables"])
tables[tables["description"].str.contains("Total Population|Median Household Income", case=False, na=False)]

Explanation of the returned fields ([IPUMS NHGIS Metadata API Documentation](https://developer.ipums.org/docs/v1/workflows/explore_metadata/nhgis/datasets/)):

- `name`: The unique identifier for the data table within its dataset.
- `description`: A short description of the data table.
- `universe`: The statistical population measured by the data table, such as persons (Total population), families (Households	), or occupied housing units.
- `nhgisCode`: The code for the data table that appears in the extract.
- `sequence`: The order in which the data table appears in the metadata API and extracts.
- `nVariables`: Number of variables included in the table.

2. **Submit an NHGIS extract request** using the IPUMS API key, specifying block groups as the geographic level and the states covering the seven MSAs.

Key variables are listed below. See the [official ACS Table IDs page](https://www.census.gov/programs-surveys/acs/data/data-tables/table-ids-explained.html#accordion-889eff65b1-item-cf09bd7a48) for more information.
- `B01003`: Total population.
- `B19013`: Median household income in the past 12 months (inflation-adjusted dollars).
- `blck_grp`: Census block group (CBG).

In [ ]:
extract = {
    "datasets": {
        dataset: {
            "dataTables": ["B01003", "B19013"], # "B01003" for total population, and "B19013" for median household income
            "geogLevels": ["blck_grp"] # Census block group (CBG) as the geographic level
        }
    },
    "geographicExtents": ["080", "120", "130", "270", "480", "530", "060"],  # the extent
    "dataFormat": "csv_no_header",
    "breakdownAndDataTypeLayout": "single_file",
    "description": "2021 ACS CBG population and household income"
}

url = "https://api.ipums.org/extracts/"
r = requests.post(url, headers={**headers, "Content-Type": "application/json"}, params={"collection": "nhgis", "version": 2}, json=extract)
result = r.json()
extract_number = result["number"]

print("Extract number:", extract_number)
print("Status:", result["status"])

3. **Check the extract status and download the completed data** from IPUMS.

In [ ]:
status_url = f"https://api.ipums.org/extracts/{extract_number}"

while True:
    status = requests.get(status_url, headers=headers, params={"collection": "nhgis", "version": 2}).json()
    if status["status"] == "completed":
        break
    print(status["status"])
    time.sleep(5)

download_url = status["downloadLinks"]["tableData"]["url"]
zip_data = requests.get(download_url, headers=headers).content

with zipfile.ZipFile(io.BytesIO(zip_data)) as z:
    print(z.namelist())
    csv_file = [f for f in z.namelist() if f.endswith(".csv")][0]
    acs = pd.read_csv(z.open(csv_file))

acs.head(2)

Key variables in `nhgis0078_ds254_20215_blck_grp.csv` are listed below:

- `GISJOIN`: Unique NHGIS geographic identifier for each Census Block Group (CBG).
- `YEAR`: Reference year of the ACS data.
- `STUSAB`: State abbreviation.
- `STATE`: State name.
- `STATEA`: State FIPS code.
- `COUNTY`: County name.
- `COUNTYA`: County FIPS code.
- `TRACTA`: Census tract code.
- `BLKGRPA`: Census block group code.
- `AON4E001`: Total population (`B01003` as the source).
- `AOQIE001`: Median household income in the past 12 months (`B19013` as the source).

4. **Filter the downloaded CBG records by state and county FIPS codes** to obtain the ACS attributes for each MSA.

In [ ]:
msa_counties = {
    "TCMA": ("27", TCMA_county_fips),
    "LA": ("06", LA_county_fips),
    "Houston": ("48", Houston_county_fips),
    "Seattle": ("53", Seattle_county_fips),
    "Miami": ("12", Miami_county_fips),
    "Atlanta": ("13", Atlanta_county_fips),
    "Denver": ("08", Denver_county_fips)
}

msa_acs = {}

for name, (state, counties) in msa_counties.items():
    msa_acs[name] = acs[
        (acs["STATEA"].astype(str).str.zfill(2) == state) &
        (acs["COUNTYA"].astype(str).str.zfill(3).isin(counties))
    ].copy()

### Data visualization

Link each msa_acs table to its corresponding CBG boundary by using `GEOID` and `GEO_ID`:

In [ ]:
def link_acs_boundary(acs_df, boundary):
    acs_df = acs_df.copy()
    boundary = boundary.copy()
    acs_df["GEOID"] = acs_df["GEO_ID"].str[-12:] # match the GEOID and GEO_ID by their structures
    boundary["GEOID"] = boundary["GEOID"].astype(str)
    return boundary.merge(acs_df, on="GEOID", how="left")

msa_boundaries = {
    "TCMA": TCMA_cbg,
    "LA": LA_cbg,
    "Houston": Houston_cbg,
    "Seattle": Seattle_cbg,
    "Miami": Miami_cbg,
    "Atlanta": Atlanta_cbg,
    "Denver": Denver_cbg
}

msa_cbg_acs = {name: link_acs_boundary(msa_acs[name], boundary) for name, boundary in msa_boundaries.items()}

TCMA_cbg_acs = msa_cbg_acs["TCMA"]
LA_cbg_acs = msa_cbg_acs["LA"]
Houston_cbg_acs = msa_cbg_acs["Houston"]
Seattle_cbg_acs = msa_cbg_acs["Seattle"]
Miami_cbg_acs = msa_cbg_acs["Miami"]
Atlanta_cbg_acs = msa_cbg_acs["Atlanta"]
Denver_cbg_acs = msa_cbg_acs["Denver"]

Verify the spatial join:

In [ ]:
for name, gdf in msa_cbg_acs.items():
    print(name, len(gdf), gdf["GEO_ID"].notna().sum())

Mapping:

In [ ]:
def map_msa_socio(gdf, name, population_col="AON4E001", income_col="AOQIE001"):
    gdf = gdf.copy()
    gdf_proj = gdf.to_crs(gdf.estimate_utm_crs())
    gdf["pop_density"] = gdf[population_col] / (gdf_proj.geometry.area / 1e6) # calculate the population density

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    gdf.plot(column="pop_density", cmap="YlOrRd", scheme="quantiles", k=5, legend=True, ax=axes[0], edgecolor="none")
    axes[0].set_title(f"{name} - Population Density")  # use pop density instead
    axes[0].axis("off")

    gdf.plot(column=income_col, cmap="YlGnBu", scheme="quantiles", k=5, legend=True, ax=axes[1], edgecolor="none")
    axes[1].set_title(f"{name} - Median Household Income")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()
    return gdf

In [ ]:
for name, gdf in msa_cbg_acs.items():
    msa_cbg_acs[name] = map_msa_socio(gdf, name)

## Summary

After running the notebook, you will have:
- CBG boundaries for each of the seven MSAs;
- 2021 ACS total population and median household income at the CBG level;
- linked GeoDataFrames combining ACS attributes with CBG geometries; and
- statistical summaries and maps for population density and median household income.

The resulting MSA-level GeoDataFrames can be exported or reused as resident-census inputs in downstream analyses.
